<a href="https://colab.research.google.com/github/amit-sw/colab_notebooks/blob/main/Evals_1_pdf_extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai google-genai pypdf pandas

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("OpenAI key loaded:", bool(os.environ.get("OPENAI_API_KEY")))
print("Gemini key loaded:", bool(os.environ.get("GEMINI_API_KEY")))

In [ ]:
from pathlib import Path

pdf_files = list(Path("/content").glob("*.pdf"))

print("PDF files found:")
for f in pdf_files:
    print("-", f.name)

PDF_PATH = "/content/Thermostats.pdf"  # change this if your uploaded file has a different name

if not Path(PDF_PATH).exists():
    raise FileNotFoundError(
        f"Could not find {PDF_PATH}. Upload the PDF to Colab or change PDF_PATH."
    )

print("Using PDF:", PDF_PATH)

In [ ]:
PROMPT = """
You are a careful product catalog extraction assistant.

The input is a thermostat product catalog PDF.

Extract product rows into structured JSON.

Return only valid JSON with this schema:

{
  "document_type": "product_catalog",
  "products": [
    {
      "category": string_or_null,
      "description": string_or_null,
      "part_number": string_or_null,
      "length": string_or_null,
      "listed_price": number_or_null,
      "currency": string_or_null,
      "page_number": number_or_null,
      "evidence": string,
      "confidence": "high" | "medium" | "low",
      "human_review_needed": boolean,
      "notes": string
    }
  ],
  "document_level_notes": string
}

Rules:
- Extract only rows where a visible part number and listed price are present.
- Do not invent part numbers.
- Do not invent prices.
- If a price is unclear, use null and mark human_review_needed as true.
- If a page is rotated, dense, or hard to read, mark confidence as "medium" or "low".
- Include page_number whenever possible.
- Include a short evidence snippet copied from the PDF.
- Use "THB" as currency if the catalog does not explicitly show another currency.
- Return only valid JSON. Do not use markdown.
"""

In [ ]:
import base64
import json
import re
from pathlib import Path

import pandas as pd
from pypdf import PdfReader, PdfWriter


def make_subset_pdf(input_pdf: str, output_pdf: str, pages: list[int]) -> str:
    """
    Create a smaller PDF with selected 1-based page numbers.
    Example: pages=[15] extracts page 15.
    """
    reader = PdfReader(input_pdf)
    writer = PdfWriter()

    for page_num in pages:
        zero_based = page_num - 1

        if zero_based < 0 or zero_based >= len(reader.pages):
            raise ValueError(f"Page {page_num} is outside the PDF range.")

        writer.add_page(reader.pages[zero_based])

    with open(output_pdf, "wb") as f:
        writer.write(f)

    return output_pdf


def extract_json_from_text(text: str) -> dict:
    """
    Handles clean JSON and occasional markdown-wrapped JSON.
    """
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()

    match = re.search(r"\{.*\}", text, re.DOTALL)

    if not match:
        print("Raw model response:")
        print(text)
        raise ValueError("No JSON object found in model response.")

    return json.loads(match.group(0))


def pdf_to_data_url(pdf_path: str) -> str:
    data = Path(pdf_path).read_bytes()
    encoded = base64.b64encode(data).decode("utf-8")
    return f"data:application/pdf;base64,{encoded}"


def save_outputs(result: dict, output_prefix: str):
    output_dir = Path("/content/outputs")
    output_dir.mkdir(exist_ok=True)

    json_path = output_dir / f"{output_prefix}.json"
    csv_path = output_dir / f"{output_prefix}.csv"

    with open(json_path, "w") as f:
        json.dump(result, f, indent=2)

    products = result.get("products", [])
    df = pd.DataFrame(products)
    df.to_csv(csv_path, index=False)

    print(f"Saved JSON: {json_path}")
    print(f"Saved CSV:  {csv_path}")

    return df, json_path, csv_path

OpenAI extractor

In [ ]:
def extract_with_openai(pdf_path: str, model: str = "gpt-4.1-mini") -> dict:
    from openai import OpenAI

    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    pdf_data_url = pdf_to_data_url(pdf_path)

    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_file",
                        "filename": Path(pdf_path).name,
                        "file_data": pdf_data_url,
                    },
                    {
                        "type": "input_text",
                        "text": PROMPT,
                    },
                ],
            }
        ],
    )

    return extract_json_from_text(response.output_text)

Gemini extractor

In [ ]:
def extract_with_gemini(pdf_path: str, model: str = "gemini-2.5-flash") -> dict:
    from google import genai

    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

    uploaded_file = client.files.upload(file=pdf_path)

    response = client.models.generate_content(
        model=model,
        contents=[
            uploaded_file,
            PROMPT,
        ],
    )

    return extract_json_from_text(response.text)

Run on one easy page first

In [ ]:
Path("/content/outputs").mkdir(exist_ok=True)

subset_pdf = make_subset_pdf(
    input_pdf=PDF_PATH,
    output_pdf="/content/outputs/page_15.pdf",
    pages=[15]
)

print("Created subset PDF:", subset_pdf)

Run with OpenAI

In [ ]:
openai_result = extract_with_openai(
    pdf_path=subset_pdf,
    model="gpt-4.1-mini"
)

openai_df, openai_json_path, openai_csv_path = save_outputs(
    openai_result,
    output_prefix="page15_openai"
)

openai_df

Run with Gemini

In [ ]:
gemini_result = extract_with_gemini(
    pdf_path=subset_pdf,
    model="gemini-2.5-flash"
)

gemini_df, gemini_json_path, gemini_csv_path = save_outputs(
    gemini_result,
    output_prefix="page15_gemini"
)

gemini_df

Download the CSV

In [ ]:
from google.colab import files

#files.download(str(openai_csv_path))

In [ ]:
#files.download(str(gemini_csv_path))

# Evals

## First we need to establish the evaluation criteria - what is important to us

In this case, we'd say that the three key fields are Part Number, Listed Price, and Currency.

In [ ]:
import csv

def load_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def compare(data, golden):
    columns = [c for c in golden[0] if c in data[0]]
    #print(f"{columns=},{golden[0]=},{data[0]=}")
    total = 0
    matches = 0
    column_results = {}
    for col in columns:
        col_matches = 0
        for row1, row2 in zip(data, golden):
            v1 = row1.get(col, "").strip()
            v2 = row2.get(col, "").strip()
            total += 1
            if v1 == v2:
                matches += 1
                col_matches += 1
        column_results[col] = {
            "matches": col_matches,
            "total": len(golden),
            "accuracy": col_matches / len(golden)
        }
    return {
        "columns": column_results,
        "overall_matches": matches,
        "overall_total": total,
        "overall_accuracy": matches / total if total else 0
    }

In [ ]:
openai_list = load_csv(str(openai_csv_path))
gemini_list = load_csv(str(gemini_csv_path))
golden_list = load_csv("/content/golden_data.csv")


result_openai = compare(openai_list, golden_list)
result_gemini = compare(gemini_list, golden_list)



In [ ]:

print("OpenAI\n")
print(json.dumps(result_openai,indent=2))
print("*****\n Gemini\n")
print(json.dumps(result_gemini,indent=2))

In [ ]:
def print_comparison_table(name1, r1, name2, r2):
    columns = r1["columns"].keys()
    print(
        f"{'Column':<20}"
        f"{name1:<15}"
        f"{name2:<15}"
    )
    print("-" * 50)
    for col in columns:
        a1 = r1["columns"][col]["accuracy"]
        a2 = r2["columns"][col]["accuracy"]
        print(
            f"{col:<20}"
            f"{a1:<15.2%}"
            f"{a2:<15.2%}"
        )
    print("-" * 50)
    print(
        f"{'OVERALL':<20}"
        f"{r1['overall_accuracy']:<15.2%}"
        f"{r2['overall_accuracy']:<15.2%}"
    )

In [ ]:
print_comparison_table('OpenAI', result_openai, 'Gemini', result_gemini)